In [1]:
import ipywidgets as widgets
import os, time

---

In [2]:
def fmt_time(seconds):
    
    if seconds is None:
        return "--"
    
    seconds = int(round(seconds))
    h, rem  = divmod(seconds, 3600)
    m, s    = divmod(rem, 60)

    if h > 0:
        return f"{h}h {m}m {s}s"
    if m > 0:
        return f"{m}m {s}s"
    return f"{s}s"

In [3]:
def write_progress(progress_file, percent):
    """
    Use this inside the slow function.
    Example:
        write_progress(progress_file, 37)
    """
    
    os.makedirs(os.path.dirname(progress_file), exist_ok=True)
    
    with open(progress_file, "a") as f:
        f.write(f"{int(percent)}\n")
        f.flush()
        os.fsync(f.fileno())

In [4]:
def update_progress(progress_file, progress_bar, percent_label, timing_label, poll_seconds=0.2):
    
    start_time        = time.monotonic()
    last_percent      = None
    last_percent_time = start_time
    last_percent_dt   = None

    while True:
        
        percent = last_percent if last_percent is not None else 0
        
        if os.path.exists(progress_file):
            
            with open(progress_file, "r") as f:
                lines = f.readlines()
            
            if lines:
                last_line = lines[-1].strip()
                
                try:
                    percent = int(last_line)
                    percent = max(0, min(100, percent))
                except ValueError:
                    pass

        now = time.monotonic()

        # Only recompute "last %" duration when the percentage actually changes
        if percent != last_percent:
            
            if last_percent is None:
                last_percent_dt = None
            else:
                dpercent = max(1, percent - last_percent)
                last_percent_dt = (now - last_percent_time) / dpercent
            
            last_percent      = percent
            last_percent_time = now

        elapsed = now - start_time
        avg_per_percent = elapsed / percent if percent > 0 else None

        progress_bar.value = percent
        percent_label.value = f"{percent}%"

        timing_label.value = (
            f"   |   total: {fmt_time(elapsed)}"
            f"   |   avg/%: {fmt_time(avg_per_percent)}"
            f"   |   last %: {fmt_time(last_percent_dt)}"
        )

        if percent >= 100:
            break

        time.sleep(poll_seconds)

---
---
---

This shows the percentage of levels we have executed.

Unfortunately, this is not a 1-1 execution time representation... early percentages may take longer, as they have more voids to merge... then agian, same happens in some od levels.

In [6]:
output_count = 6

progress_bar  = widgets.IntProgress(min=0, max=100, value=0, description="Progress:")
percent_label = widgets.Label(value="0%")
timing_label  = widgets.Label(value="   |   total: 0s   |   avg/%: --   |   last %: --")

hbox = widgets.HBox([progress_bar, percent_label, timing_label])
display(hbox)

notebook_parameters = [{"progress_file": "./progress/" + str(output_count) + ".log"}]
progress_file = notebook_parameters[0]["progress_file"]

update_progress(progress_file = progress_file,
                progress_bar  = progress_bar,
                percent_label = percent_label,
                timing_label  = timing_label,
                poll_seconds  = 0.2)

---
---
---